# Prompt Testing Demo cho Sudoku-Bench

## 1. Cài đặt thư viện

Chạy cell dưới nếu máy chưa có thư viện Google GenAI.


In [ ]:
# Nếu chưa cài thư viện, bỏ comment dòng dưới:
# !pip install google-genai

## 2. Khai báo Gemini API theo cách của TV1

In [1]:
import os
import json
import re
from datetime import datetime
from google import genai
from google.genai import types

os.environ["GOOGLE_CLOUD_PROJECT"] = "project-038ccd57-3d62-4aac-8b5"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

client = genai.Client(
    vertexai=True,
    project=os.environ["GOOGLE_CLOUD_PROJECT"],
    location=os.environ["GOOGLE_CLOUD_LOCATION"]
)

def call_gemini(prompt, model_name="gemini-2.5-flash"):
    config = types.GenerateContentConfig(
        temperature=0,
    )

    response = client.models.generate_content(
        model=model_name,
        contents=prompt,
        config=config
    )

    return response.text

## 3. Khai báo puzzle

Puzzle được biểu diễn dưới dạng text giống tinh thần Sudoku-Bench:

- Initial Board
- Rules
- Visual Elements

In [2]:
PUZZLE_NAME = "Prime Cage 4x4 Demo"
GRID_SIZE = 4

initial_board = "................"

rules = """Each row and column contains the digits 1-4 once each.
Digits in a cage sum to a prime number but no two cages sum to the same prime.
Digits in a cage MAY repeat."""

visual_elements = """- killer cage: r1c2 r1c3 r2c3
- killer cage: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- killer cage: r2c2 r3c2 r4c1 r4c2
- killer cage: r1c1 r2c1 r3c1"""

cages = [
    ["r1c2", "r1c3", "r2c3"],
    ["r1c4", "r2c4", "r3c3", "r3c4", "r4c3", "r4c4"],
    ["r2c2", "r3c2", "r4c1", "r4c2"],
    ["r1c1", "r2c1", "r3c1"],
]

print("Puzzle:", PUZZLE_NAME)
print("\nInitial Board:")
print(initial_board)
print("\nRules:")
print(rules)
print("\nVisual Elements:")
print(visual_elements)


Puzzle: Prime Cage 4x4 Demo

Initial Board:
................

Rules:
Each row and column contains the digits 1-4 once each.
Digits in a cage sum to a prime number but no two cages sum to the same prime.
Digits in a cage MAY repeat.

Visual Elements:
- killer cage: r1c2 r1c3 r2c3
- killer cage: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- killer cage: r2c2 r3c2 r4c1 r4c2
- killer cage: r1c1 r2c1 r3c1


## 4. Hàm hỗ trợ xử lý board và tọa độ

In [3]:
def board_string_to_grid(board_str, n=4):
    if len(board_str) != n * n:
        raise ValueError(f"Board phải có {n*n} ký tự.")

    grid = []
    for i in range(0, len(board_str), n):
        row = []
        for ch in board_str[i:i+n]:
            row.append(0 if ch == "." else int(ch))
        grid.append(row)
    return grid

def grid_to_board_string(grid):
    chars = []
    for row in grid:
        for value in row:
            chars.append("." if value == 0 else str(value))
    return "".join(chars)

def coord_to_index(coord):
    r_part, c_part = coord.split("c")
    r = int(r_part.replace("r", "")) - 1
    c = int(c_part) - 1
    return r, c

def get_cell(grid, coord):
    r, c = coord_to_index(coord)
    return grid[r][c]

def set_cell(grid, row, col, value):
    # row, col nhập theo kiểu 1-based cho dễ đọc
    new_grid = [r[:] for r in grid]
    new_grid[row - 1][col - 1] = value
    return new_grid

initial_grid = board_string_to_grid(initial_board, GRID_SIZE)
initial_grid


[[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]

## 5. Validator cho puzzle

Validator kiểm tra:

1. Grid có đúng kích thước 4x4 không.
2. Mỗi hàng chứa 1-4 đúng một lần.
3. Mỗi cột chứa 1-4 đúng một lần.
4. Tổng mỗi cage là số nguyên tố.
5. Không có hai cage nào có cùng tổng nguyên tố.

In [4]:
def is_prime(n):
    if n < 2:
        return False
    for x in range(2, int(n ** 0.5) + 1):
        if n % x == 0:
            return False
    return True

def validate_rows_and_columns(grid):
    expected = {1, 2, 3, 4}

    if len(grid) != 4 or any(len(row) != 4 for row in grid):
        return False, "Grid không phải 4x4."

    for i, row in enumerate(grid, start=1):
        if set(row) != expected:
            return False, f"Hàng {i} không chứa đúng các số 1-4 một lần."

    for c in range(4):
        col = {grid[r][c] for r in range(4)}
        if col != expected:
            return False, f"Cột {c+1} không chứa đúng các số 1-4 một lần."

    return True, "Rows and columns OK."

def validate_cages(grid, cages):
    cage_sums = []

    for idx, cage in enumerate(cages, start=1):
        values = [get_cell(grid, coord) for coord in cage]
        cage_sum = sum(values)
        cage_sums.append(cage_sum)

        if not is_prime(cage_sum):
            return False, f"Cage {idx} có values={values}, sum={cage_sum}, không phải số nguyên tố."

    if len(cage_sums) != len(set(cage_sums)):
        return False, f"Các cage có tổng bị trùng: {cage_sums}"

    return True, f"Cages OK. Cage sums = {cage_sums}"

def validate_puzzle(grid):
    checks = [
        validate_rows_and_columns(grid),
        validate_cages(grid, cages),
    ]

    is_valid = all(ok for ok, msg in checks)
    messages = [msg for ok, msg in checks]
    return is_valid, messages


## 6. Brute force tìm nghiệm đúng để kiểm tra validator

Phần này không dùng LLM.  
Ta sinh tất cả Latin square 4x4 rồi lọc theo luật cage.

Mục đích:

- Kiểm tra puzzle có nghiệm không.
- Có ground truth để so sánh kết quả model.


In [5]:
import itertools

def generate_latin_4x4():
    perms = list(itertools.permutations([1, 2, 3, 4]))
    for rows in itertools.product(perms, repeat=4):
        grid = [list(row) for row in rows]
        ok, _ = validate_rows_and_columns(grid)
        if ok:
            yield grid

all_solutions = []
for grid in generate_latin_4x4():
    ok, _ = validate_puzzle(grid)
    if ok:
        all_solutions.append(grid)

print("Số nghiệm hợp lệ tìm được:", len(all_solutions))

for i, sol in enumerate(all_solutions[:5], start=1):
    print(f"\nSolution {i}:")
    for row in sol:
        print(row)


Số nghiệm hợp lệ tìm được: 1

Solution 1:
[4, 2, 1, 3]
[1, 3, 2, 4]
[2, 4, 3, 1]
[3, 1, 4, 2]


## 7. Hàm parse JSON từ response của Gemini


Mục tiêu là lấy được:

```json
{
  "grid": [[...], [...], [...], [...]]
}
```

hoặc với multi-step:

```json
{
  "row": 1,
  "col": 2,
  "value": 3
}
```

In [6]:
def extract_json_from_response(text):
    if text is None:
        raise ValueError("Response text là None. Có thể model bị chặn, hết token, hoặc API không trả nội dung.")

    text = text.strip()

    # Xóa markdown code fence nếu có
    text = re.sub(r"^```json\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # Tìm JSON object đầu tiên
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("Không tìm thấy JSON object trong response.")

    return json.loads(match.group(0))


## 8. Tạo phần mô tả puzzle dùng chung cho tất cả prompt

Phần này giúp các prompt thống nhất cùng một input.


In [8]:
def build_puzzle_description(board_str=None):
    if board_str is None:
        board_str = initial_board

    return f"""
Puzzle name: {PUZZLE_NAME}

Initial Board:
{board_str}

Rules:
{rules}

Visual Elements:
{visual_elements}

Coordinate system:
- r1c1 means row 1 column 1.
- The board is 4x4.
- Empty cells are represented by dot '.'.
""".strip()

print(build_puzzle_description())


Puzzle name: Prime Cage 4x4 Demo

Initial Board:
................

Rules:
Each row and column contains the digits 1-4 once each.
Digits in a cage sum to a prime number but no two cages sum to the same prime.
Digits in a cage MAY repeat.

Visual Elements:
- killer cage: r1c2 r1c3 r2c3
- killer cage: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- killer cage: r2c2 r3c2 r4c1 r4c2
- killer cage: r1c1 r2c1 r3c1

Coordinate system:
- r1c1 means row 1 column 1.
- The board is 4x4.
- Empty cells are represented by dot '.'.


# 9. Prompt


Ta sẽ tạo 4 loại prompt khác nhau về chiến lược hoạt động:

1. Baseline Prompt
2. Chain-of-Thought Prompt
3. Verification Prompt
4. Multi-step Prompt

## Prompt 1 — Baseline


Baseline là prompt đơn giản nhất:

- Đưa puzzle cho model.
- Yêu cầu giải toàn bộ.
- Trả JSON.

Prompt này dùng làm mốc so sánh.

In [20]:
def build_baseline_prompt():
    return f"""
You are solving a Sudoku variant puzzle.

{build_puzzle_description()}

Important constraints:
- Each row must contain digits 1, 2, 3, 4 exactly once.
- Each column must contain digits 1, 2, 3, 4 exactly once.
- Digits inside a cage MAY repeat.
- Every cage sum must be a PRIME number.
- No two cages may have the same prime sum.

Cages are defined in this exact order:
- cage_1: r1c2 r1c3 r2c3
- cage_2: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- cage_3: r2c2 r3c2 r4c1 r4c2
- cage_4: r1c1 r2c1 r3c1

Important invalid example:
If cage_2 has values [3, 2, 3, 1, 1, 4],
then:

3 + 2 + 3 + 1 + 1 + 4 = 14

14 is NOT prime.

So that grid must be rejected.

Another invalid example:
If cage sums are:

[7, 17, 11, 7]

this is INVALID because cage sums must all be unique.

Critical instruction:
Compute cage_sums DIRECTLY from your final returned grid.
Do NOT guess cage_sums.
The cage_sums must exactly match the grid values in the same cage order.

Task:
Solve the puzzle completely.

Before returning the answer:
1. Compute the final grid.
2. Compute cage_1 sum from the grid.
3. Compute cage_2 sum from the grid.
4. Compute cage_3 sum from the grid.
5. Compute cage_4 sum from the grid.
6. Check every sum is prime.
7. Check all cage sums are unique.
8. Only then return JSON.

Return ONLY valid JSON in this exact format:
{{
  "grid": [
    [1, 2, 3, 4],
    [3, 4, 1, 2],
    [2, 1, 4, 3],
    [4, 3, 2, 1]
  ],
  "cage_sums": [5, 17, 11, 7]
}}

Do not include markdown.
Do not include explanation outside JSON.
""".strip()

## Prompt 2 — Chain-of-Thought / Reasoning Prompt

Prompt này yêu cầu model suy luận trước khi đưa đáp án.

Trong thực nghiệm, ta vẫn yêu cầu output cuối cùng chứa JSON để parse được.

In [38]:
def build_cot_prompt():
    return f"""
You are solving a Sudoku variant puzzle.

{build_puzzle_description()}

Important constraints:
- Each row must contain digits 1, 2, 3, 4 exactly once.
- Each column must contain digits 1, 2, 3, 4 exactly once.
- There are NO 2x2 box constraints.
- Digits inside a cage MAY repeat.
- Every cage sum must be a prime number.
- No two cages may have the same prime sum.

Cages in exact order:
- cage_1: r1c2 r1c3 r2c3
- cage_2: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- cage_3: r2c2 r3c2 r4c1 r4c2
- cage_4: r1c1 r2c1 r3c1

CoT procedure:
1. Propose a candidate grid.
2. Compute cage sums from that grid.
3. Reject if:
   - any row/column is invalid
   - any cage sum is not prime
   - any cage sums are duplicated
4. Try again.
5. Use AT MOST 3 attempts.

Error codes:
- row_col_error
- non_prime
- duplicate_sum
- none

Return ONLY valid JSON in this exact format:
{{
  "attempts": [
    {{
      "id": 1,
      "grid": [
        [1, 2, 3, 4],
        [2, 1, 4, 3],
        [3, 4, 1, 2],
        [4, 3, 2, 1]
      ],
      "sums": [9, 12, 10, 6],
      "status": "rejected",
      "error": "non_prime"
    }},
    {{
      "id": 2,
      "grid": [
        [4, 2, 1, 3],
        [1, 3, 2, 4],
        [2, 4, 3, 1],
        [3, 1, 4, 2]
      ],
      "sums": [5, 17, 11, 7],
      "status": "accepted",
      "error": "none"
    }}
  ],
  "final_grid": [
    [4, 2, 1, 3],
    [1, 3, 2, 4],
    [2, 4, 3, 1],
    [3, 1, 4, 2]
  ],
  "final_sums": [5, 17, 11, 7],
  "is_valid_by_model": true
}}

Important:
- The JSON example is FORMAT ONLY.
- Maximum 3 attempts.
- Do not include markdown.
- Do not include explanation outside JSON.
""".strip()

cot_prompt = build_cot_prompt()
print(cot_prompt)


You are solving a Sudoku variant puzzle.

Puzzle name: Prime Cage 4x4 Demo

Initial Board:
................

Rules:
Each row and column contains the digits 1-4 once each.
Digits in a cage sum to a prime number but no two cages sum to the same prime.
Digits in a cage MAY repeat.

Visual Elements:
- killer cage: r1c2 r1c3 r2c3
- killer cage: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- killer cage: r2c2 r3c2 r4c1 r4c2
- killer cage: r1c1 r2c1 r3c1

Coordinate system:
- r1c1 means row 1 column 1.
- The board is 4x4.
- Empty cells are represented by dot '.'.

Important constraints:
- Each row must contain digits 1, 2, 3, 4 exactly once.
- Each column must contain digits 1, 2, 3, 4 exactly once.
- There are NO 2x2 box constraints.
- Digits inside a cage MAY repeat.
- Every cage sum must be a prime number.
- No two cages may have the same prime sum.

Cages in exact order:
- cage_1: r1c2 r1c3 r2c3
- cage_2: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- cage_3: r2c2 r3c2 r4c1 r4c2
- cage_4: r1c1 r2c1 r3c1

CoT procedure

Prompt này yêu cầu model **tự kiểm tra lại** trước khi trả lời:

- Kiểm tra hàng
- Kiểm tra cột
- Kiểm tra cage sum là số nguyên tố
- Kiểm tra các cage không trùng tổng

In [59]:
def build_verification_prompt():
    return f"""
You are solving a Sudoku variant puzzle.

{build_puzzle_description()}

Important constraints:
- Each row must contain digits 1, 2, 3, 4 exactly once.
- Each column must contain digits 1, 2, 3, 4 exactly once.
- There are NO 2x2 box constraints.
- Digits inside a cage MAY repeat.
- Every cage sum must be a prime number.
- No two cages may have the same prime sum.

Cages in exact order:
- cage_1: r1c2 r1c3 r2c3
- cage_2: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- cage_3: r2c2 r3c2 r4c1 r4c2
- cage_4: r1c1 r2c1 r3c1

Verification procedure:
Before returning the final grid, verify:
1. Rows are valid.
2. Columns are valid.
3. Cage sums are computed directly from the final grid.
4. Every cage sum is prime.
5. Cage sums are all unique.

Return ONLY valid JSON in this exact format:
{{
  "grid": [
    [4, 2, 1, 3],
    [1, 3, 2, 4],
    [2, 4, 3, 1],
    [3, 1, 4, 2]
  ],
  "verification": {{
    "rows_valid": true,
    "columns_valid": true,
    "cage_sums": [5, 17, 11, 7],
    "all_cage_sums_prime": true,
    "all_cage_sums_unique": true,
    "is_valid_by_model": true
  }}
}}

Important:
- The JSON example is FORMAT ONLY.
- Compute cage_sums directly from your returned grid.
- Do not guess cage_sums.
- Do not include markdown.
- Do not include explanation outside JSON.
""".strip()

verification_prompt = build_verification_prompt()
print(verification_prompt)


You are solving a Sudoku variant puzzle.

Puzzle name: Prime Cage 4x4 Demo

Initial Board:
................

Rules:
Each row and column contains the digits 1-4 once each.
Digits in a cage sum to a prime number but no two cages sum to the same prime.
Digits in a cage MAY repeat.

Visual Elements:
- killer cage: r1c2 r1c3 r2c3
- killer cage: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- killer cage: r2c2 r3c2 r4c1 r4c2
- killer cage: r1c1 r2c1 r3c1

Coordinate system:
- r1c1 means row 1 column 1.
- The board is 4x4.
- Empty cells are represented by dot '.'.

Important constraints:
- Each row must contain digits 1, 2, 3, 4 exactly once.
- Each column must contain digits 1, 2, 3, 4 exactly once.
- There are NO 2x2 box constraints.
- Digits inside a cage MAY repeat.
- Every cage sum must be a prime number.
- No two cages may have the same prime sum.

Cages in exact order:
- cage_1: r1c2 r1c3 r2c3
- cage_2: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4
- cage_3: r2c2 r3c2 r4c1 r4c2
- cage_4: r1c1 r2c1 r3c1

Verification 

## Prompt 3 — Multi-step Prompt

Multi-step khác với 3 prompt trên:

- Không yêu cầu giải toàn bộ trong 1 lần.
- Mỗi vòng chỉ yêu cầu model trả về **một ô chắc chắn**.
- Sau đó notebook cập nhật board và hỏi tiếp.


In [18]:
def grid_to_pretty_text(grid):
    lines = []
    for row in grid:
        line = " ".join("." if v == 0 else str(v) for v in row)
        lines.append(line)
    return "\n".join(lines)

In [17]:
def build_multistep_prompt(current_grid, step_id=1):
    current_board_text = grid_to_pretty_text(current_grid)

    return f"""
You are solving a 4x4 Sudoku variant puzzle in multi-step mode.

Puzzle name: {PUZZLE_NAME}

Current Board:
{current_board_text}

Rules:
{rules}

Visual Elements:
{visual_elements}

Important:
- Only fill empty cells marked with '.'.
- Do not change already filled cells.
- Return at least ONE next digit placement.
- The system will update the board and continue.
- Rows and columns must contain digits 1-4 exactly once.
- Each cage sum must be prime.
- No two cages may have the same prime sum.
- Digits inside a cage MAY repeat.

Current step: {step_id}

Return ONLY valid JSON:
{{
  "placements": [
    {{"row": 1, "col": 2, "value": 3}}
  ]
}}
""".strip()

# 10. Chạy thực nghiệm single-shot

Phần này chạy 2 prompt single-shot:

1. Baseline
2. CoT

In [ ]:
single_shot_prompts = {
    "baseline": build_baseline_prompt(),
    "cot": build_cot_prompt(),
}

single_shot_results = []

## Baseline

In [ ]:
def run_single_shot_experiment_basline(prompt_name, prompt, model_name="gemini-2.5-flash"):
    print("=" * 80)
    print("Running:", prompt_name)
    print("=" * 80)

    result = {
        "prompt_name": prompt_name,
        "model_name": model_name,
        "prompt": prompt,
        "raw_response": None,
        "parsed": None,
        "predicted_grid": None,
        "is_valid": False,
        "messages": [],
        "error": None,
    }

    try:
        raw_response = call_gemini(prompt, model_name=model_name)
        result["raw_response"] = raw_response

        print("\nRaw response:")
        print(raw_response)

        parsed = extract_json_from_response(raw_response)
        result["parsed"] = parsed

        predicted_grid = parsed["grid"]
        result["predicted_grid"] = predicted_grid

        model_cage_sums = parsed.get("cage_sums", None)
        result["model_cage_sums"] = model_cage_sums

        is_valid, messages = validate_puzzle(predicted_grid)
        result["is_valid"] = is_valid
        result["messages"] = messages

        print("\nPredicted grid:")
        for row in predicted_grid:
            print(row)

        print("\nModel cage_sums:")
        print(model_cage_sums)

        print("\nValid:", is_valid)
        print("Messages:")
        for msg in messages:
            print("-", msg)

    except Exception as e:
        result["error"] = str(e)
        print("ERROR:", e)

    return result


In [ ]:
result = run_single_shot_experiment_basline("baseline", build_baseline_prompt())

Running: baseline

Raw response:
```json
{
  "grid": [
    [4, 2, 1, 3],
    [1, 3, 2, 4],
    [2, 4, 3, 1],
    [3, 1, 4, 2]
  ],
  "cage_sums": [5, 17, 11, 7]
}
```

Predicted grid:
[4, 2, 1, 3]
[1, 3, 2, 4]
[2, 4, 3, 1]
[3, 1, 4, 2]

Model cage_sums:
[5, 17, 11, 7]

Valid: True
Messages:
- Rows and columns OK.
- Cages OK. Cage sums = [5, 17, 11, 7]


## Cot

In [41]:
def run_single_shot_experiment_cot(prompt_name, prompt, model_name="gemini-2.5-flash"):
    print("=" * 80)
    print("Running:", prompt_name)
    print("=" * 80)

    result = {
        "prompt_name": prompt_name,
        "model_name": model_name,
        "prompt": prompt,
        "raw_response": None,
        "parsed": None,
        "attempts": None,
        "final_grid": None,
        "final_cage_sums": None,
        "is_valid": False,
        "messages": [],
        "error": None,
    }

    try:
        raw_response = call_gemini(prompt, model_name=model_name)
        result["raw_response"] = raw_response

        print("\nRaw response:")
        print(raw_response)

        parsed = extract_json_from_response(raw_response)
        result["parsed"] = parsed

        attempts = parsed.get("attempts", [])
        final_grid = parsed["final_grid"]
        final_cage_sums = parsed.get("final_sums", None)

        result["attempts"] = attempts
        result["final_grid"] = final_grid
        result["final_sums"] = final_cage_sums

        print("\nAttempts from model:")
        for attempt in attempts:
            print("-" * 50)
            print("Attempt ID:", attempt.get("id"))
            print("Status:", attempt.get("status"))
            print("Reason:", attempt.get("reason"))
            print("Model cage_sums:", attempt.get("sums"))

            attempt_grid = attempt.get("grid")
            if attempt_grid:
                print("Attempt grid:")
                for row in attempt_grid:
                    print(row)

                py_valid, py_messages = validate_puzzle(attempt_grid)
                print("Python validator:", py_valid)
                for msg in py_messages:
                    print("  -", msg)

        print("\nFinal grid:")
        for row in final_grid:
            print(row)

        print("\nFinal cage sums:")
        print(final_cage_sums)

        is_valid, messages = validate_puzzle(final_grid)
        result["is_valid"] = is_valid
        result["messages"] = messages

        print("\nFinal Python validator:", is_valid)
        for msg in messages:
            print("-", msg)

    except Exception as e:
        result["error"] = str(e)
        print("ERROR:", e)

    return result

In [39]:
run_single_shot_experiment_cot("cot", build_cot_prompt())

Running: cot

Raw response:
{
  "attempts": [
    {
      "id": 1,
      "grid": [
        [1, 2, 3, 4],
        [2, 1, 4, 3],
        [3, 4, 1, 2],
        [4, 3, 2, 1]
      ],
      "sums": [9, 13, 12, 6],
      "status": "rejected",
      "error": "non_prime"
    },
    {
      "id": 2,
      "grid": [
        [4, 2, 1, 3],
        [1, 3, 2, 4],
        [2, 4, 3, 1],
        [3, 1, 4, 2]
      ],
      "sums": [5, 17, 11, 7],
      "status": "accepted",
      "error": "none"
    }
  ],
  "final_grid": [
    [4, 2, 1, 3],
    [1, 3, 2, 4],
    [2, 4, 3, 1],
    [3, 1, 4, 2]
  ],
  "final_sums": [5, 17, 11, 7],
  "is_valid_by_model": true
}

Attempts from model:
--------------------------------------------------
Attempt ID: 1
Status: rejected
Reason: None
Model cage_sums: [9, 13, 12, 6]
Attempt grid:
[1, 2, 3, 4]
[2, 1, 4, 3]
[3, 4, 1, 2]
[4, 3, 2, 1]
Python validator: False
  - Rows and columns OK.
  - Cage 1 có values=[2, 3, 4], sum=9, không phải số nguyên tố.
---------------------

{'prompt_name': 'cot',
 'model_name': 'gemini-2.5-flash',
 'prompt': 'You are solving a Sudoku variant puzzle.\n\nPuzzle name: Prime Cage 4x4 Demo\n\nInitial Board:\n................\n\nRules:\nEach row and column contains the digits 1-4 once each.\nDigits in a cage sum to a prime number but no two cages sum to the same prime.\nDigits in a cage MAY repeat.\n\nVisual Elements:\n- killer cage: r1c2 r1c3 r2c3\n- killer cage: r1c4 r2c4 r3c3 r3c4 r4c3 r4c4\n- killer cage: r2c2 r3c2 r4c1 r4c2\n- killer cage: r1c1 r2c1 r3c1\n\nCoordinate system:\n- r1c1 means row 1 column 1.\n- The board is 4x4.\n- Empty cells are represented by dot \'.\'.\n\nImportant constraints:\n- Each row must contain digits 1, 2, 3, 4 exactly once.\n- Each column must contain digits 1, 2, 3, 4 exactly once.\n- There are NO 2x2 box constraints.\n- Digits inside a cage MAY repeat.\n- Every cage sum must be a prime number.\n- No two cages may have the same prime sum.\n\nCages in exact order:\n- cage_1: r1c2 r1c3 r2c3\n- ca

# 11. Chạy thực nghiệm multi-step

Phần này cho model đi từng bước.

Cách đánh giá trong notebook:

- Sau mỗi placement, notebook cập nhật board.
- Nếu placement làm board không còn khả năng dẫn đến nghiệm hợp lệ, dừng.
- Nếu fill đủ board và validate đúng, xem như solve thành công.

Để kiểm tra một placement có còn hợp lệ không, ta so với tập nghiệm brute force đã tìm được ở phần trước.

In [23]:
def extract_json_from_multistep_response(text):
    if text is None:
        raise ValueError("Response is None")

    key = '"placements"'
    key_pos = text.find(key)
    if key_pos == -1:
        raise ValueError("Không tìm thấy key placements trong response.")

    start = text.rfind("{", 0, key_pos)
    if start == -1:
        raise ValueError("Không tìm thấy dấu { trước placements.")

    depth = 0
    in_string = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if escape:
            escape = False
            continue

        if ch == "\\":
            escape = True
            continue

        if ch == '"':
            in_string = not in_string
            continue

        if not in_string:
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    json_text = text[start:i+1]
                    return json.loads(json_text)

    raise ValueError("JSON placements object bị thiếu dấu đóng }.")

def grid_matches_solution_partial(grid, solution):
    for r in range(4):
        for c in range(4):
            if grid[r][c] != 0 and grid[r][c] != solution[r][c]:
                return False
    return True

def has_possible_solution(grid, solutions):
    return any(grid_matches_solution_partial(grid, sol) for sol in solutions)

def is_grid_full(grid):
    return all(value != 0 for row in grid for value in row)

def print_grid(grid):
    for row in grid:
        print(row)

def run_multistep_experiment(max_steps=16, model_name="gemini-2.5-flash"):
    current_grid = board_string_to_grid(initial_board, GRID_SIZE)

    result = {
        "prompt_name": "multi_step",
        "model_name": model_name,
        "steps": [],
        "final_grid": None,
        "is_valid": False,
        "messages": [],
        "error": None,
    }

    for step_id in range(1, max_steps + 1):
        prompt = build_multistep_prompt(current_grid, step_id)

        print("=" * 80)
        print(f"Multi-step - Step {step_id}")
        print("=" * 80)
        print("Current grid:")
        print_grid(current_grid)

        try:
            raw_response = call_gemini(prompt, model_name=model_name)
            print("\nRaw response:")
            print(raw_response)

            parsed = extract_json_from_multistep_response(raw_response)
            placements = parsed.get("placements", [])

            if not placements:
                step_record = {
                    "step": step_id,
                    "prompt": prompt,
                    "raw_response": raw_response,
                    "parsed": parsed,
                    "accepted": [],
                    "rejected": [],
                    "message": "Model không trả placement nào."
                }
                result["steps"].append(step_record)
                print("Stop:", step_record["message"])
                break

            step_record = {
                "step": step_id,
                "prompt": prompt,
                "raw_response": raw_response,
                "parsed": parsed,
                "accepted": [],
                "rejected": [],
                "message": "",
            }

            for move in placements:
                row = int(move["row"])
                col = int(move["col"])
                value = int(move["value"])

                move_record = {
                    "row": row,
                    "col": col,
                    "value": value,
                    "accepted": False,
                    "reason": "",
                }

                # Check range
                if not (1 <= row <= 4 and 1 <= col <= 4 and 1 <= value <= 4):
                    move_record["reason"] = "Ngoài phạm vi hợp lệ."
                    step_record["rejected"].append(move_record)
                    continue

                # Check empty cell
                if current_grid[row - 1][col - 1] != 0:
                    move_record["reason"] = "Ô đã được điền trước đó."
                    step_record["rejected"].append(move_record)
                    continue

                # Chỉ update board, không check ground truth ở đây
                current_grid = set_cell(current_grid, row, col, value)
                move_record["accepted"] = True
                move_record["reason"] = "Accepted by format/current-board rules."
                step_record["accepted"].append(move_record)

                print(f"Accepted placement: r{row}c{col} = {value}")

            result["steps"].append(step_record)

            # Nếu model trả toàn move bị reject thì dừng
            if len(step_record["accepted"]) == 0:
                step_record["message"] = "Không có placement hợp lệ theo format/current board."
                print("Stop:", step_record["message"])
                break

            # Nếu board đầy thì validate cuối cùng
            if is_grid_full(current_grid):
                is_valid, messages = validate_puzzle(current_grid)
                result["final_grid"] = current_grid
                result["is_valid"] = is_valid
                result["messages"] = messages

                print("\nFinal grid:")
                print_grid(current_grid)
                print("\nValid:", is_valid)
                for msg in messages:
                    print("-", msg)
                break

        except Exception as e:
            result["error"] = str(e)
            print("ERROR:", e)
            break

    if result["final_grid"] is None:
        result["final_grid"] = current_grid
        if is_grid_full(current_grid):
            is_valid, messages = validate_puzzle(current_grid)
        else:
            is_valid, messages = False, ["Board chưa hoàn thành."]
        result["is_valid"] = is_valid
        result["messages"] = messages

    return result

Chạy multi-step

In [24]:
multi_step_result = run_multistep_experiment(max_steps=16)

Multi-step - Step 1
Current grid:
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 0, 0, 0]

Raw response:
The first step is to analyze the cages and their possible sums, considering the standard Sudoku rules for rows and columns.

**1. Analyze Cage C4 (r1c1, r2c1, r3c1):**
*   This cage consists of 3 cells in column 1: r1c1, r2c1, r3c1.
*   According to Sudoku rules, the digits in column 1 (r1c1, r2c1, r3c1, r4c1) must be 1, 2, 3, 4 exactly once.
*   Therefore, the digits in C4 (r1c1, r2c1, r3c1) must be distinct.
*   The sum of the digits in C4 must be a prime number.
*   Possible sums for 3 distinct digits from {1, 2, 3, 4}:
    *   {1, 2, 3} sum = 6 (not prime)
    *   {1, 2, 4} sum = 7 (prime!)
    *   {1, 3, 4} sum = 8 (not prime)
    *   {2, 3, 4} sum = 9 (not prime)
*   The only combination of 3 distinct digits from {1, 2, 3, 4} that sums to a prime number is {1, 2, 4}, which sums to 7.
*   Therefore, the digits in C4 (r1c1, r2c1, r3c1) must be 1, 2, and 4 in some order, and their su